# SAE Probing and Concept Analysis
Experimenting with different thresholds and selectivity ranges

In [1]:
import torch
import json
import numpy as np
import tiktoken
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import glob, os

from sparse_auto_encoder import SparseAutoencoder
from utils.model import load_GPT_model
from sae_probing.extract_latent_activations import exract_latent_activations
from sae_probing.filter_selective_neurons import find_selective_neurons
from sae_probing.neuron_concept_assoc import calculate_neuron_to_concept_assoc
from sae_probing.neuron_concept_mapping import build_neuron_concept_map

### 1. Setup and Data Loading

In [2]:
device = "cpu"

In [3]:
model = load_GPT_model(path="model_896_14_8_256.pth", device=device)

In [4]:
sae_1 = SparseAutoencoder(input_dim=896, hidden_dim=2688).to(device)
sae_1.load_state_dict(torch.load("sae_models/sae_layer1.pth", map_location=torch.device('cpu')))
sae_1.eval();

sae_2 = SparseAutoencoder(input_dim=896, hidden_dim=2688).to(device)
sae_2.load_state_dict(torch.load("sae_models/sae_layer2.pth", map_location=torch.device('cpu')))
sae_2.eval();

sae_3 = SparseAutoencoder(input_dim=896, hidden_dim=3584).to(device)
sae_3.load_state_dict(torch.load("sae_models/sae_layer3.pth", map_location=torch.device('cpu')))
sae_3.eval();

sae_4 = SparseAutoencoder(input_dim=896, hidden_dim=3584).to(device)
sae_4.load_state_dict(torch.load("sae_models/sae_layer4.pth", map_location=torch.device('cpu')))
sae_4.eval();

sae_5 = SparseAutoencoder(input_dim=896, hidden_dim=3584).to(device)
sae_5.load_state_dict(torch.load("sae_models/sae_layer5.pth", map_location=torch.device('cpu')))
sae_5.eval();

sae_6 = SparseAutoencoder(input_dim=896, hidden_dim=4480).to(device)
sae_6.load_state_dict(torch.load("sae_models/sae_layer6.pth", map_location=torch.device('cpu')))
sae_6.eval();

sae_7 = SparseAutoencoder(input_dim=896, hidden_dim=4480).to(device)
sae_7.load_state_dict(torch.load("sae_models/sae_layer7.pth", map_location=torch.device('cpu')))
sae_7.eval();

sae_8 = SparseAutoencoder(input_dim=896, hidden_dim=4480).to(device)
sae_8.load_state_dict(torch.load("sae_models/sae_layer8.pth", map_location=torch.device('cpu')))
sae_8.eval();

In [5]:
latents_l1 = exract_latent_activations(model, sae_1, layer=1)
latents_l2 = exract_latent_activations(model, sae_2, layer=2)
latents_l3 = exract_latent_activations(model, sae_3, layer=3)
latents_l4 = exract_latent_activations(model, sae_4, layer=4)
latents_l5 = exract_latent_activations(model, sae_5, layer=5)
latents_l6 = exract_latent_activations(model, sae_6, layer=6)
latents_l7 = exract_latent_activations(model, sae_7, layer=7)
latents_l8 = exract_latent_activations(model, sae_8, layer=8)

✅ Saved sae_probing\activations/latent_activations_l1.pt with latents shape torch.Size([665, 2688]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l2.pt with latents shape torch.Size([665, 2688]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l3.pt with latents shape torch.Size([665, 3584]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l4.pt with latents shape torch.Size([665, 3584]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l5.pt with latents shape torch.Size([665, 3584]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l6.pt with latents shape torch.Size([665, 4480]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l7.pt with latents shape torch.Size([665, 4480]) and 665 ids.
✅ Saved sae_probing\activations/latent_activations_l8.pt with latents shape torch.Size([665, 4480]) and 665 ids.


### 2. Layer-wise Concept Mapping - Different filtering configurations

In [6]:
def map_layer_neurons(
    layer,
    activation_threshold=5.0,
    min_count=5,
    max_count=150,
    out_dir_name="results"):

    ROOT_DIR = "sae_probing"
    OUT_DIR = os.path.join(ROOT_DIR, out_dir_name)
    
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(f"{OUT_DIR}/mappings", exist_ok=True)
    os.makedirs(f"{OUT_DIR}/analysis", exist_ok=True)

    try:
        find_selective_neurons(layer=layer, activation_threshold=activation_threshold, 
                               min_count=min_count, max_count=max_count, 
                               root_dir=ROOT_DIR, out_dir=OUT_DIR)
        
        calculate_neuron_to_concept_assoc(layer=layer, threshold=activation_threshold, root_dir=ROOT_DIR, out_dir=OUT_DIR);
    
        mappings = build_neuron_concept_map(layer=layer, base_dir=OUT_DIR)
        print(f"✅ Done for layer {layer}")
        print('='*10)
        
        return mappings.head()
    except:
        print(f"Error generating mappings for layer {layer}")
        print('='*10)
        return None

In [7]:
from sae_probing.neuron_mapping_analysis import cross_layers_mapping_analysis

def map_all_layers(activation_threshold=5.0, min_sentences_count=5, max_sentences_count=150):
    
    OUT_DIR_NAME = f"thr_{'_'.join(str(activation_threshold).split('.'))}_min_{min_sentences_count}_max_{max_sentences_count}"
    base_dir = f"sae_probing/{OUT_DIR_NAME}"

    # Map neurons per layer
    map_layer_neurons(layer=1, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=2, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=3, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=4, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=5, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=6, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=7, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)
    map_layer_neurons(layer=8, activation_threshold=activation_threshold, min_count=min_sentences_count, max_count=max_sentences_count, out_dir_name=OUT_DIR_NAME)

    # Merge all layers associations
    pattern = os.path.join(base_dir, "mappings", "neuron_label_assoc_l*.csv")
    files = sorted(glob.glob(pattern))
    
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            # make sure each file has a 'layer' column
            if "layer" not in df.columns:
                # try to parse layer number from filename
                layer_num = int(os.path.basename(f).split("_l")[-1].split(".")[0])
                df["layer"] = layer_num
            dfs.append(df)
        except:
            print(f"ERROR empty csv file: {f}")
    
    merged = pd.concat(dfs, ignore_index=True)
    out_path = os.path.join(base_dir, "mappings", "neuron_label_assoc_all_layers.csv")
    merged.to_csv(out_path, index=False)
    
    print(f"✅ Merged {len(files)} files into {out_path}, total rows: {len(merged)}")

    # Merge all layers primary-secondary mappings
    pattern = os.path.join(base_dir, "mappings", "neuron_concept_primary_secondary_l*.csv")
    files = sorted(glob.glob(pattern))
    
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # make sure each file has a 'layer' column
        if "layer" not in df.columns:
            # try to parse layer number from filename
            layer_num = int(os.path.basename(f).split("_l")[-1].split(".")[0])
            df["layer"] = layer_num
        dfs.append(df)
    
    merged = pd.concat(dfs, ignore_index=True)
    out_path = os.path.join(base_dir, "mappings", "neuron_concept_primary_secondary_all_layers.csv")
    merged.to_csv(out_path, index=False)
    
    print(f"✅ Merged {len(files)} files into {out_path}, total rows: {len(merged)}")

    # run full cross-layer analysis
    cross_layers_mapping_analysis(base_dir=f"sae_probing/{OUT_DIR_NAME}")

---

In [8]:
map_all_layers(activation_threshold=3.0, min_sentences_count=5, max_sentences_count=150)

Layer 1: N=665 tokensets, H=2688 neurons.
Window [5, 150], thresh=3.0. Selective found: 16
-> IDs saved to: sae_probing\thr_3_0_min_5_max_150\mappings\selective_neuron_ids_l1.pt
✅ Associations table saved: sae_probing\thr_3_0_min_5_max_150\mappings\neuron_label_assoc_l1.csv (176 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_3_0_min_5_max_150\mappings\neuron_concept_primary_secondary_l1.csv
✅ Done for layer 1
Layer 2: N=665 tokensets, H=2688 neurons.
Window [5, 150], thresh=3.0. Selective found: 50
-> IDs saved to: sae_probing\thr_3_0_min_5_max_150\mappings\selective_neuron_ids_l2.pt
✅ Associations table saved: sae_probing\thr_3_0_min_5_max_150\mappings\neuron_label_assoc_l2.csv (550 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_3_0_min_5_max_150\mappings\neuron_concept_primary_secondary_l2.csv
✅ Done for layer 2
Layer 3: N=665 tokensets, H=3584 neurons.
Window [5, 150], thresh=3.0. Selective found: 85
-> IDs saved to: sae_probing\thr_3_0_min_5_max_150\mappings\sel

In [9]:
map_all_layers(activation_threshold=5.0, min_sentences_count=0, max_sentences_count=200)

Layer 1: N=665 tokensets, H=2688 neurons.
Window [0, 200], thresh=5.0. Selective found: 2688
-> IDs saved to: sae_probing\thr_5_0_min_0_max_200\mappings\selective_neuron_ids_l1.pt
✅ Associations table saved: sae_probing\thr_5_0_min_0_max_200\mappings\neuron_label_assoc_l1.csv (29568 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_5_0_min_0_max_200\mappings\neuron_concept_primary_secondary_l1.csv
✅ Done for layer 1
Layer 2: N=665 tokensets, H=2688 neurons.
Window [0, 200], thresh=5.0. Selective found: 2688
-> IDs saved to: sae_probing\thr_5_0_min_0_max_200\mappings\selective_neuron_ids_l2.pt
✅ Associations table saved: sae_probing\thr_5_0_min_0_max_200\mappings\neuron_label_assoc_l2.csv (29568 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_5_0_min_0_max_200\mappings\neuron_concept_primary_secondary_l2.csv
✅ Done for layer 2
Layer 3: N=665 tokensets, H=3584 neurons.
Window [0, 200], thresh=5.0. Selective found: 3583
-> IDs saved to: sae_probing\thr_5_0_min_0_max_200\ma

In [10]:
map_all_layers(activation_threshold=7.0, min_sentences_count=5, max_sentences_count=150)

Layer 1: N=665 tokensets, H=2688 neurons.
Window [5, 150], thresh=7.0. Selective found: 0
-> IDs saved to: sae_probing\thr_7_0_min_5_max_150\mappings\selective_neuron_ids_l1.pt
✅ Associations table saved: sae_probing\thr_7_0_min_5_max_150\mappings\neuron_label_assoc_l1.csv (0 rows)
Error generating mappings for layer 1
Layer 2: N=665 tokensets, H=2688 neurons.
Window [5, 150], thresh=7.0. Selective found: 5
-> IDs saved to: sae_probing\thr_7_0_min_5_max_150\mappings\selective_neuron_ids_l2.pt
✅ Associations table saved: sae_probing\thr_7_0_min_5_max_150\mappings\neuron_label_assoc_l2.csv (55 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_7_0_min_5_max_150\mappings\neuron_concept_primary_secondary_l2.csv
✅ Done for layer 2
Layer 3: N=665 tokensets, H=3584 neurons.
Window [5, 150], thresh=7.0. Selective found: 13
-> IDs saved to: sae_probing\thr_7_0_min_5_max_150\mappings\selective_neuron_ids_l3.pt
✅ Associations table saved: sae_probing\thr_7_0_min_5_max_150\mappings\neuron_la

In [11]:
map_all_layers(activation_threshold=3.0, min_sentences_count=10, max_sentences_count=100)

Layer 1: N=665 tokensets, H=2688 neurons.
Window [10, 100], thresh=3.0. Selective found: 5
-> IDs saved to: sae_probing\thr_3_0_min_10_max_100\mappings\selective_neuron_ids_l1.pt
✅ Associations table saved: sae_probing\thr_3_0_min_10_max_100\mappings\neuron_label_assoc_l1.csv (55 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_3_0_min_10_max_100\mappings\neuron_concept_primary_secondary_l1.csv
✅ Done for layer 1
Layer 2: N=665 tokensets, H=2688 neurons.
Window [10, 100], thresh=3.0. Selective found: 22
-> IDs saved to: sae_probing\thr_3_0_min_10_max_100\mappings\selective_neuron_ids_l2.pt
✅ Associations table saved: sae_probing\thr_3_0_min_10_max_100\mappings\neuron_label_assoc_l2.csv (242 rows)
✅ Primary/secondary mapping saved: sae_probing\thr_3_0_min_10_max_100\mappings\neuron_concept_primary_secondary_l2.csv
✅ Done for layer 2
Layer 3: N=665 tokensets, H=3584 neurons.
Window [10, 100], thresh=3.0. Selective found: 51
-> IDs saved to: sae_probing\thr_3_0_min_10_max_100\mapp

---

In [12]:
from sae_probing.compare_all_runs import compare_all_runs

compare_all_runs()

Scanning runs in sae_probing
Loaded run: thr_3_0_min_10_max_100 (717 rows)
Loaded run: thr_3_0_min_20_max_100 (440 rows)
Loaded run: thr_3_0_min_5_max_150 (1178 rows)
Loaded run: thr_5_0_min_0_max_200 (29561 rows)
Loaded run: thr_5_0_min_5_max_150 (521 rows)
Loaded run: thr_7_0_min_5_max_150 (268 rows)

Found runs: ['thr_3_0_min_10_max_100', 'thr_3_0_min_20_max_100', 'thr_3_0_min_5_max_150', 'thr_5_0_min_0_max_200', 'thr_5_0_min_5_max_150', 'thr_7_0_min_5_max_150']

=== Comparing thr_3_0_min_10_max_100 vs thr_3_0_min_20_max_100 ===


C:\Users\IuG_Lap1\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:284: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesB = pd.factorize(list(zip(pairsB["primary_concept"], pairsB["secondary_concept"])))[0]



=== Comparing thr_3_0_min_10_max_100 vs thr_3_0_min_5_max_150 ===


C:\Users\IuG_Lap1\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:284: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesB = pd.factorize(list(zip(pairsB["primary_concept"], pairsB["secondary_concept"])))[0]



=== Comparing thr_3_0_min_10_max_100 vs thr_5_0_min_0_max_200 ===


...

C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:284: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesB = pd.factorize(list(zip(pairsB["primary_concept"], pairsB["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_conce


=== Comparing thr_3_0_min_5_max_150 vs thr_5_0_min_5_max_150 ===

=== Comparing thr_3_0_min_5_max_150 vs thr_7_0_min_5_max_150 ===


C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:284: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesB = pd.factorize(list(zip(pairsB["primary_concept"], pairsB["secondary_concept"])))[0]
C:\Users\IuG_Lap1\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that i


=== Comparing thr_5_0_min_0_max_200 vs thr_5_0_min_5_max_150 ===

=== Comparing thr_5_0_min_0_max_200 vs thr_7_0_min_5_max_150 ===

=== Comparing thr_5_0_min_5_max_150 vs thr_7_0_min_5_max_150 ===
Saved metrics summary to: sae_probing\run_metrics_summary.csv
Saved dominant-overlap summary to: sae_probing\run_dominant_overlap_summary.csv


C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:284: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesB = pd.factorize(list(zip(pairsB["primary_concept"], pairsB["secondary_concept"])))[0]
C:\Users\IuG_Lap1\Desktop\development\GPTAndPrejudice\sae_probing\compare_all_runs.py:283: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codesA = pd.factorize(list(zip(pairsA["primary_concept"], pairsA["secondary_conce